In [7]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

In [8]:
from pathlib import Path
import sys
import json
import contextlib
import io
import numpy as np
import pandas as pd
import optuna
from icdn import PanelSchema
from icdn.data.splits import TemporalSplitter

ROOT = Path.cwd() if Path.cwd().name != "notebooks" else Path.cwd().parent
sys.path.insert(0, str(ROOT))

from src.benchmarks.features import ICDNFeaturePipeline
from src.benchmarks.demand_mlp import DemandMLPPipeline

DATASETS = {
    "walmart": {
        "path": ROOT / "data" / "M5-walmart" / "panel" / "m5_icdn_panel.parquet",
        "out": ROOT / "data" / "M5-walmart" / "panel" / "mlp-opt",
        "schema": PanelSchema(category="category"),
    },
    "one_c": {
        "path": ROOT / "data" / "predict-future-sales-1c" / "panel" / "1c_icdn_panel.parquet",
        "out": ROOT / "data" / "predict-future-sales-1c" / "panel" / "mlp-opt",
        "schema": PanelSchema(category="category"),
    },
}

N_FOLDS = 3
MIN_TRAIN_FRAC = 0.5
N_TRIALS = 1
SEED = 42
HIDDEN_CHOICES = {"256_128_64": (256, 128, 64), "128_64_32": (128, 64, 32), "128_64": (128, 64)}


def load_panel(spec):
    panel = pd.read_parquet(spec["path"])
    return panel[(panel["price"] > 0) & (panel["units"] > 0)].copy()


def featurize(spec, train_raw, val_raw):
    feats = ICDNFeaturePipeline(schema=spec["schema"])
    train = feats.fit(train_raw).transform(train_raw)
    val = feats.transform_val(val_raw)
    return train, val, feats.control_cols


def prepare_folds(spec):
    panel = load_panel(spec)
    splitter = TemporalSplitter(period_col="week_id")
    folds = splitter.expanding_splits(panel, n_folds=N_FOLDS, min_train_frac=MIN_TRAIN_FRAC)
    return [featurize(spec, tr, va) for tr, va in folds]


def suggest_mlp(trial):
    hidden_key = trial.suggest_categorical("hidden", list(HIDDEN_CHOICES))
    return dict(
        hidden=HIDDEN_CHOICES[hidden_key],
        dropout=trial.suggest_float("dropout", 0.05, 0.40),
        lr=trial.suggest_float("lr", 5e-4, 3e-3, log=True),
        act="gelu",
        weight_decay=1e-5,
        d_store=16,
        huber_delta=1.0,
        n_epochs=80,
        es_patience=15,
        seed=SEED,
    )


def mae_one(train, val, control_cols, params):
    mlp = DemandMLPPipeline(control_cols, **params)
    with contextlib.redirect_stdout(io.StringIO()):
        metrics, _ = mlp.run(train, val)
    return float(metrics["mae_val"])


def objective(trial, prepared):
    params = suggest_mlp(trial)
    maes = []
    for k, (train, val, controls) in enumerate(prepared):
        mae = mae_one(train, val, controls, params)
        maes.append(mae)
        trial.set_user_attr(f"fold{k}_mae", mae)
        trial.report(float(np.mean(maes)), k)
        if trial.should_prune():
            raise optuna.TrialPruned()
    return float(np.mean(maes))


def dump_best(study, out_dir):
    best = dict(study.best_params)
    best["hidden"] = list(HIDDEN_CHOICES[best["hidden"]])
    out_dir.mkdir(parents=True, exist_ok=True)
    (out_dir / "best_params.json").write_text(json.dumps(best, indent=2))
    study.trials_dataframe().to_csv(out_dir / "optuna_trials.csv", index=False)
    print("best MAE", study.best_value)
    print("best params", best)
    print("wrote", out_dir / "best_params.json")
    return best


def run_study(name, spec):
    print(f"\n=== {name} Optuna ===")
    prepared = prepare_folds(spec)
    out_dir = spec["out"]
    out_dir.mkdir(parents=True, exist_ok=True)
    study = optuna.create_study(
        study_name=f"mlp_{name}",
        storage=f"sqlite:///{out_dir / 'optuna.db'}",
        load_if_exists=True,
        direction="minimize",
        sampler=optuna.samplers.TPESampler(seed=SEED),
        pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=0),
    )
    study.optimize(lambda t: objective(t, prepared), n_trials=N_TRIALS, gc_after_trial=True)
    return dump_best(study, out_dir)

In [9]:
best_by_dataset = {}
for name, spec in DATASETS.items():
    best_by_dataset[name] = run_study(name, spec)


=== walmart Optuna ===


[I 2026-08-25 12:07:13,552] A new study created in RDB with name: mlp_walmart
[I 2026-08-25 12:07:16,348] Trial 0 finished with value: 0.5564771890640259 and parameters: {'hidden': '128_64_32', 'dropout': 0.25953046946896285, 'lr': 0.0006612658646612707}. Best is trial 0 with value: 0.5564771890640259.


best MAE 0.5564771890640259
best params {'hidden': [128, 64, 32], 'dropout': 0.25953046946896285, 'lr': 0.0006612658646612707}
wrote /home/thebigmonster/Github/nn-elasticity-additional-work/data/M5-walmart/panel/mlp-opt/best_params.json

=== one_c Optuna ===


[I 2026-08-25 12:07:17,229] A new study created in RDB with name: mlp_one_c
[I 2026-08-25 12:07:18,249] Trial 0 finished with value: 0.3688977261384328 and parameters: {'hidden': '128_64_32', 'dropout': 0.25953046946896285, 'lr': 0.0006612658646612707}. Best is trial 0 with value: 0.3688977261384328.


best MAE 0.3688977261384328
best params {'hidden': [128, 64, 32], 'dropout': 0.25953046946896285, 'lr': 0.0006612658646612707}
wrote /home/thebigmonster/Github/nn-elasticity-additional-work/data/predict-future-sales-1c/panel/mlp-opt/best_params.json
